# 🚢 Titanic Survival Prediction
### Stacking Ensemble: Random Forest + XGBoost + Gradient Boosting + SVM

---

## Project Overview
Predict which passengers survived the Titanic disaster using the classic Kaggle dataset.  
This project demonstrates **advanced feature engineering**, **stacking ensembles**, and **Optuna hyperparameter tuning** to push classification accuracy.

**Dataset:** [Kaggle Titanic Competition](https://www.kaggle.com/c/titanic)  
**Task:** Binary Classification (Survived: 0 or 1)  
**Best Accuracy:** ~82%+ on Kaggle leaderboard

---

## Table of Contents
1. [Install & Import Libraries](#1)
2. [Load & Explore Data](#2)
3. [Feature Engineering](#3)
4. [Preprocessing Pipeline](#4)
5. [Stacking Ensemble Model](#5)
6. [Optuna Hyperparameter Tuning](#6)
7. [Evaluation Metrics](#7)
8. [Generate Kaggle Submission](#8)

## 1. Install & Import Libraries

In [ ]:
# Install required packages
!pip install xgboost optuna --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
import optuna

print('All libraries imported successfully!')

## 2. Load & Explore Data

In [ ]:
# Load datasets (download from Kaggle: https://www.kaggle.com/c/titanic/data)
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')
gender_submission = pd.read_csv('gender_submission.csv')

print('Train shape:            ', train.shape)   # (891, 12)
print('Test shape:             ', test.shape)    # (418, 11)
print('Gender submission shape:', gender_submission.shape)
train.head()

In [ ]:
# Exploratory Data Analysis
print('=== Missing Values ===')
print(train.isnull().sum())

print('\n=== Survival Rate ===')
print(train['Survived'].value_counts(normalize=True).round(2))

print('\n=== Survival by Gender ===')
print(train.groupby('Sex')['Survived'].mean().round(2))

print('\n=== Survival by Class ===')
print(train.groupby('Pclass')['Survived'].mean().round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Survival count
train['Survived'].value_counts().plot(kind='bar', ax=axes[0], color=['#e74c3c','#2ecc71'])
axes[0].set_title('Survival Count')
axes[0].set_xticklabels(['Died', 'Survived'], rotation=0)

# Survival by gender
train.groupby('Sex')['Survived'].mean().plot(kind='bar', ax=axes[1], color=['#3498db','#e91e8c'])
axes[1].set_title('Survival Rate by Gender')
axes[1].set_ylabel('Survival Rate')
axes[1].set_xticklabels(['Female', 'Male'], rotation=0)

# Survival by class
train.groupby('Pclass')['Survived'].mean().plot(kind='bar', ax=axes[2], color=['#f39c12','#9b59b6','#1abc9c'])
axes[2].set_title('Survival Rate by Class')
axes[2].set_xticklabels(['1st', '2nd', '3rd'], rotation=0)

plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
def engineer_features(df):
    df = df.copy()

    # ── Title extraction from Name ────────────────────────────
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(
        ['Lady','Countess','Capt','Col','Don','Dr',
         'Major','Rev','Sir','Jonkheer','Dona'], 'Rare')
    df['Title'] = df['Title'].replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})

    # ── Family features ───────────────────────────────────────
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone']    = (df['FamilySize'] == 1).astype(int)
    df['FamilyGroup'] = pd.cut(df['FamilySize'],
                                bins=[0,1,4,20],
                                labels=['Alone','Small','Large'])

    # ── Smart Age fill using Title median ─────────────────────
    age_medians = df.groupby('Title')['Age'].transform('median')
    df['Age']   = df['Age'].fillna(age_medians)

    # ── Interaction features ──────────────────────────────────
    df['Age_x_Class']   = df['Age'] * df['Pclass']
    df['Fare']          = df['Fare'].fillna(df['Fare'].median())
    df['FarePerPerson'] = df['Fare'] / df['FamilySize']

    # ── Ticket frequency ──────────────────────────────────────
    ticket_counts    = df['Ticket'].map(df['Ticket'].value_counts())
    df['TicketFreq'] = ticket_counts

    # ── Deck from Cabin ───────────────────────────────────────
    df['Deck'] = df['Cabin'].str[0].fillna('Unknown')

    return df

train = engineer_features(train)
test  = engineer_features(test)

print('New features added:')
print([c for c in train.columns if c not in ['PassengerId','Survived','Name','Ticket','Cabin']])

## 4. Preprocessing Pipeline

In [ ]:
# Define feature groups
FEATURES = ['Pclass','Sex','Age','SibSp','Parch','Fare','Embarked',
            'Title','FamilySize','IsAlone','FamilyGroup',
            'Age_x_Class','FarePerPerson','TicketFreq','Deck']

numeric_features  = ['Age','SibSp','Parch','Fare','FamilySize',
                     'Age_x_Class','FarePerPerson','TicketFreq','IsAlone']
categorical_features = ['Pclass','Sex','Embarked','Title','FamilyGroup','Deck']

# Build preprocessing pipeline
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer,  numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Prepare X, y
X = train[FEATURES]
y = train['Survived']
test_ids = test['PassengerId']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Fit & transform
X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc   = preprocessor.transform(X_val)
X_processed  = preprocessor.fit_transform(X)        # full set for CV
X_test_proc  = preprocessor.transform(test[FEATURES])

print(f'Train set: {X_train_proc.shape}, Val set: {X_val_proc.shape}')

## 5. Stacking Ensemble Model

In [ ]:
# ── Base models (Layer 1) ─────────────────────────────────────
base_models = [
    ('rf',  RandomForestClassifier(
                n_estimators=300, max_depth=6,
                min_samples_split=4, random_state=42)),
    ('xgb', XGBClassifier(
                n_estimators=300, max_depth=4,
                learning_rate=0.05, subsample=0.8,
                colsample_bytree=0.8, random_state=42,
                eval_metric='logloss')),
    ('gb',  GradientBoostingClassifier(
                n_estimators=300, max_depth=4,
                learning_rate=0.05, subsample=0.8,
                random_state=42)),
    ('svm', SVC(kernel='rbf', C=1.0, probability=True, random_state=42))
]

# ── Meta model (Layer 2) ──────────────────────────────────────
meta_model = LogisticRegression(max_iter=1000)

# ── Stacking Classifier ───────────────────────────────────────
stacked_model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5,
    passthrough=True
)

# ── Cross-validate ────────────────────────────────────────────
scores = cross_val_score(stacked_model, X_processed, y, cv=5, scoring='accuracy')
print(f'Stacked Ensemble CV Accuracy: {scores.mean()*100:.2f}% ± {scores.std()*100:.2f}%')

## 6. Optuna Hyperparameter Tuning (XGBoost)

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0, 0.5)
    }
    model  = XGBClassifier(**params, random_state=42, eval_metric='logloss')
    scores = cross_val_score(model, X_processed, y, cv=5, scoring='accuracy')
    return scores.mean()

# Run 100 trials
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

print('Best parameters:', study.best_params)
print(f'Best CV accuracy: {study.best_value*100:.2f}%')

In [ ]:
# Train final model with best params
best_xgb = XGBClassifier(**study.best_params, random_state=42, eval_metric='logloss')
best_xgb.fit(
    X_train_proc,
    y_train,
    eval_set=[(X_train_proc, y_train), (X_val_proc, y_val)],
    verbose=False,
)

evals_result = best_xgb.evals_result()
train_loss = evals_result['validation_0']['logloss']
val_loss = evals_result['validation_1']['logloss']

y_pred = best_xgb.predict(X_val_proc)
print(f'Validation Accuracy: {accuracy_score(y_val, y_pred)*100:.2f}%')

In [ ]:
from sk_autod import diagnose

report = diagnose(train_loss, val_loss)

print('SK-AutoD diagnosis:')
print(report.summary())

## 7. Evaluation Metrics

In [ ]:
print('=== Classification Report ===')
print(classification_report(y_val, y_pred, target_names=['Died','Survived']))

# Confusion matrix
cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Died','Survived'],
            yticklabels=['Died','Survived'])
plt.title('Confusion Matrix — Titanic Survival Prediction')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (from the best XGBoost model)
feat_names = (
    numeric_features +
    list(preprocessor.named_transformers_['cat']
         .named_steps['onehot'].get_feature_names_out(categorical_features))
)

importance_df = pd.DataFrame({
    'Feature':   feat_names,
    'Importance': best_xgb.feature_importances_
}).sort_values('Importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df, y='Feature', x='Importance', palette='viridis')
plt.title('Top 15 Feature Importances (XGBoost)')
plt.tight_layout()
plt.show()

## 8. Generate Kaggle Submission

In [ ]:
# Retrain on full training data before submitting
best_xgb.fit(X_processed, y)
final_predictions = best_xgb.predict(X_test_proc)

submission = pd.DataFrame({
    'PassengerId': test_ids,
    'Survived':    final_predictions
})

submission.to_csv('submission.csv', index=False)
print('submission.csv saved!')
print(f'Predicted survivors: {final_predictions.sum()} / {len(final_predictions)}')
submission.head(10)